# GlassNet - look inside a from-scratch model

This notebook loads one of our **published** checkpoints with `numpy` only. No PyTorch, no
transformers, no API keys. You print the parameter count, the next-token guesses and the named
concept neurons that fire.

**Read this first:** the model is experimental. It reaches 37.37% held-out next-token accuracy
(by position) with a **32 token** window. It cannot hold a conversation and it cannot follow
instructions. This notebook is for looking inside it, not for using it as an assistant.

In [ ]:
!pip -q install huggingface_hub safetensors numpy

In [ ]:
import json, re, numpy as np
from huggingface_hub import hf_hub_download
from safetensors.numpy import load_file

REPO = 'blueprint-ai-hq-main/glassnet-d512-ext6-37pct'
ck = hf_hub_download(REPO, 'model.safetensors')
vocab_path = hf_hub_download(REPO, 'labels.json')
W = load_file(ck)
words = json.load(open(vocab_path))
print('tensors:', sorted(W.keys()))
print('vocabulary:', len(words), 'words')

In [ ]:
# the forward pass, in full: emb -> named concepts -> answer + tied readout
D, V = W['emb'].shape
K = W['concept.W'].shape[1]
WINDOW = W['dmix'].shape[0] // D
def forward(ids):
    ids = list(ids)[-WINDOW:]
    ids = [0] * (WINDOW - len(ids)) + ids          # the trainer left-pads short contexts
    x = W['emb'][np.asarray(ids)].reshape(1, WINDOW * D)
    h = 1 / (1 + np.exp(-(x @ W['concept.W'] + W['concept.b'])))   # concept layer
    lg = h @ W['answer.W'] + W['answer.b'] + np.maximum(x @ W['dmix'], 0) @ W['emb'].T
    return lg[0], h[0]
n = sum(v.size for v in W.values())
print(f'{n:,} parameters, D={D}, concepts={K}, window={WINDOW}')

In [ ]:
stoi = {w: i for i, w in enumerate(words)}
def enc(t):
    return [stoi.get(x, stoi.get('<unk>', 0)) for x in re.findall(r"[A-Za-z']+|[0-9]+|[^\sA-Za-z0-9]", t.lower())]

prompt = 'Human: hello , how are you ?\n\nAssistant:'
lg, h = forward(enc(prompt))
top = np.argsort(-lg)[:10]
print('prompt:', prompt.replace('\n', ' | '))
for i in top:
    print(f'  {words[i]!r:>14}  logit {lg[i]:7.3f}')

In [ ]:
# which named concept neurons fire, and how strong
strong = np.argsort(-h)[:10]
print('mean activation %.4f, %d of %d concepts above 0.5' % (h.mean(), int((h > 0.5).sum()), K))
for j in strong:
    print(f'  concept #{j:02d}  activation {h[j]:.3f}')

In [ ]:
# honest closure: what this model can and cannot do
print('37.37% held-out next-token accuracy (by position). Window 32 tokens.')
print('It has never been evaluated on questions. Do not expect answers.')
print('Everything it was trained on, every failed experiment, and every number is in')
print('https://github.com/blueprint-ai-hq-git/glassnet')